In [1]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

# 기온 데이터 불러오기
temperature_df = pd.read_csv(
    PROCESSED_DIR / "기상_일별_최고최저기온_정제.csv"
)

# 관측소별 기본 정보 추출
station_df = (
    temperature_df[
        ["지점", "지점명", "시도"]
    ]
    .drop_duplicates()
    .sort_values("지점")
    .reset_index(drop=True)
)

print("관측소 수:", len(station_df))
display(station_df.head(10))

관측소 수: 97


,지점,지점명,시도
0,90,속초,강원특별자치도
1,93,북춘천,강원특별자치도
2,95,철원,강원특별자치도
3,98,동두천,경기도
4,99,파주,경기도
5,100,대관령,강원특별자치도
6,101,춘천,강원특별자치도
7,102,백령도,인천광역시
8,104,북강릉,강원특별자치도
9,105,강릉,강원특별자치도


In [2]:
print("지점번호 중복:", station_df["지점"].duplicated().sum())

print("지점번호 결측:", station_df["지점"].isna().sum())

print("지점명 결측:", station_df["지점명"].isna().sum())

print("시도 결측:", station_df["시도"].isna().sum())

print("\n시도별 관측소 수")
display(
    station_df.groupby("시도")["지점"]
    .count()
    .sort_values(ascending=False)
)

지점번호 중복: 0
지점번호 결측: 0
지점명 결측: 0
시도 결측: 0

시도별 관측소 수


시도
강원특별자치도    14
경상남도       14
경상북도       14
전라남도       13
전북특별자치도    10
충청북도        6
충청남도        6
경기도         5
제주특별자치도     4
인천광역시       3
부산광역시       2
대구광역시       1
광주광역시       1
서울특별시       1
대전광역시       1
세종특별자치시     1
울산광역시       1
Name: 지점, dtype: int64

In [3]:
station_info_path = (
    ROOT_DIR / "data" / "raw" / "META_관측지점정보.csv"
)

# CSV 인코딩 확인을 위해 우선 UTF-8-SIG로 시도
try:
    station_info_df = pd.read_csv(
        station_info_path,
        encoding="utf-8-sig"
    )
except UnicodeDecodeError:
    station_info_df = pd.read_csv(
        station_info_path,
        encoding="cp949"
    )

print("데이터 크기:", station_info_df.shape)
print("\n컬럼 목록:")
print(station_info_df.columns.tolist())

display(station_info_df.head())

데이터 크기: (148, 13)

컬럼 목록:
['지점', '시작일', '종료일', '지점명', '지점주소', '관리관서', '위도', '경도', '노장해발고도(m)', '기압계(관측장비지상높이(m))', '기온계(관측장비지상높이(m))', '풍속계(관측장비지상높이(m))', '강우계(관측장비지상높이(m))']


,지점,시작일,종료일,지점명,지점주소,관리관서,위도,경도,노장해발고도(m),기압계(관측장비지상높이(m)),기온계(관측장비지상높이(m)),풍속계(관측장비지상높이(m)),강우계(관측장비지상높이(m))
0,90,1968-01-01,NaN,속초,강원특별자치도 고성군 토성면 봉포리,속초기상대(90),38.2509,128.5647,17.53,18.73,1.7,10.0,1.4
1,93,2016-10-01,NaN,북춘천,강원특별자치도 춘천시 신북읍 산천리,춘천기상대(101),37.9474,127.7544,95.78,96.78,1.5,10.0,1.4
2,95,1988-01-01,NaN,철원,강원특별자치도 철원군 갈말읍 군탄리,춘천기상대(101),38.1479,127.3042,155.48,156.98,1.8,13.0,1.5
3,98,1998-02-01,NaN,동두천,경기도 동두천시 생연동,수도권기상청(119),37.9019,127.0607,115.62,116.74,1.7,10.0,1.0
4,99,2013-10-22,NaN,파주,경기도 파주시 문산읍 운천리,수도권기상청(119),37.8859,126.7665,30.59,31.99,1.7,10.0,1.0


In [4]:
# 지점번호 자료형 통일
station_df["지점"] = pd.to_numeric(
    station_df["지점"], errors="coerce"
).astype("Int64")

station_info_df["지점"] = pd.to_numeric(
    station_info_df["지점"], errors="coerce"
).astype("Int64")

# 기존 97개 관측소 번호
target_stations = set(station_df["지점"].dropna())

# 기상청 자료에서 기존 관측소만 추출
matched_info = station_info_df[
    station_info_df["지점"].isin(target_stations)
].copy()

print("기존 관측소 수:", len(target_stations))
print("매칭된 행 수:", len(matched_info))
print("매칭된 고유 지점 수:", matched_info["지점"].nunique())

print("\n기존 관측소 중 매칭되지 않은 지점:")
print(sorted(target_stations - set(matched_info["지점"])))

print("\n중복 지점번호:")
print(
    matched_info[
        matched_info.duplicated("지점", keep=False)
    ].sort_values("지점")
)

기존 관측소 수: 97
매칭된 행 수: 139
매칭된 고유 지점 수: 97

기존 관측소 중 매칭되지 않은 지점:
[]

중복 지점번호:
      지점         시작일         종료일  지점명                      지점주소         관리관서  \
4     99  2013-10-22         NaN   파주           경기도 파주시 문산읍 운천리  수도권기상청(119)   
5     99  2001-12-07  2013-10-22   문산           경기도 파주시 문산읍 운천리    파주기상대(99)   
6    100  2006-11-07         NaN  대관령  (산지)강원특별자치도 평창군 대관령면 횡계리  대관령기상대(100)   
7    100  1971-07-15  2006-11-07  대관령      (산지)강원도 평창군 대관령면 횡계리          NaN   
9    102  2018-05-01         NaN  백령도         인천광역시 옹진군 백령면 진촌리  백령도기상대(102)   
..   ...         ...         ...  ...                       ...          ...   
133  276  2010-09-01  2020-12-04  청송군              경상북도 청송군 청송읍   안동기상대(136)   
138  283  2020-11-04         NaN  경주시               경상북도 경주시 탑동   포항기상대(138)   
139  283  2010-08-06  2020-11-04  경주시               경상북도 경주시 탑동   포항기상대(138)   
140  284  2011-03-04         NaN   거창          경상남도 거창군 거창읍 정장리   울산기상대(152)   
141  284  1972-01-24  2011-03-04   거창      

In [5]:
# 날짜 형식 변환
matched_info["시작일"] = pd.to_datetime(
    matched_info["시작일"], errors="coerce"
)

matched_info["종료일"] = pd.to_datetime(
    matched_info["종료일"], errors="coerce"
)

# 분석 기간
analysis_start = pd.Timestamp("2021-01-01")
analysis_end = pd.Timestamp("2025-12-31")

# 분석 기간과 운영 기간이 겹치는 행 추출
station_period_df = matched_info[
    (matched_info["시작일"] <= analysis_end) &
    (
        matched_info["종료일"].isna() |
        (matched_info["종료일"] >= analysis_start)
    )
].copy()

print("기존 매칭 행:", len(matched_info))
print("분석 기간과 겹치는 행:", len(station_period_df))
print("고유 관측소 수:", station_period_df["지점"].nunique())

기존 매칭 행: 139
분석 기간과 겹치는 행: 102
고유 관측소 수: 97


In [6]:
duplicate_stations = (
    station_period_df[
        station_period_df.duplicated("지점", keep=False)
    ]
    .sort_values(["지점", "시작일"])
)

print("분석 기간 중복 관측소 수:")
print(duplicate_stations["지점"].nunique())

display(
    duplicate_stations[
        ["지점", "지점명", "시작일", "종료일",
         "위도", "경도", "지점주소"]
    ]
)

분석 기간 중복 관측소 수:
4


,지점,지점명,시작일,종료일,위도,경도,지점주소
63,181,서청주(예),2022-01-21,2023-05-01,36.6397,127.3969,충청북도 청주시흥덕구 강내면 학천리
62,181,서청주,2023-05-01,NaT,36.6399,127.3846,충청북도 청주시흥덕구 강내면 학천리
77,211,인제,1971-12-01,2025-07-30,38.0599,128.1671,강원특별자치도 인제군 인제읍 남북리
76,211,인제,2025-07-30,NaT,38.0599,128.1681,강원특별자치도 인제군 인제읍 남북리
83,217,정선군,2010-08-06,2023-07-01,37.3815,128.6459,강원특별자치도 정선군 정선읍 북실리
82,217,정선군,2023-07-01,2025-12-17,37.3773,128.6735,강원특별자치도 정선군 정선읍 애산리
81,217,정선군,2025-12-17,NaT,37.3807,128.6731,강원특별자치도 정선군 정선읍 애산리
116,259,강진군,2009-11-10,2021-02-25,34.6289,126.7672,전라남도 강진군 강진읍 남포리
115,259,강진군,2021-02-25,NaT,34.6446,126.7841,전남광주통합특별시 강진군 군동면 호계리


In [7]:
import numpy as np

# 중복된 관측소만 추출
changed_stations = (
    station_period_df[
        station_period_df.duplicated("지점", keep=False)
    ]
    .sort_values(["지점", "시작일"])
    .copy()
)

def calculate_distance(lat1, lon1, lat2, lon2):
    """두 좌표 사이의 거리(km) 계산"""
    R = 6371

    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


movement_results = []

for station_id, group in changed_stations.groupby("지점"):

    group = group.sort_values("시작일")

    for i in range(len(group) - 1):
        old = group.iloc[i]
        new = group.iloc[i + 1]

        distance = calculate_distance(
            old["위도"], old["경도"],
            new["위도"], new["경도"]
        )

        movement_results.append({
            "지점": station_id,
            "관측소": new["지점명"],
            "변경일": new["시작일"].date(),
            "이전_위도": old["위도"],
            "이전_경도": old["경도"],
            "변경_위도": new["위도"],
            "변경_경도": new["경도"],
            "이동거리_km": round(distance, 3)
        })

movement_df = pd.DataFrame(movement_results)

display(movement_df)

,지점,관측소,변경일,이전_위도,이전_경도,변경_위도,변경_경도,이동거리_km
0,181,서청주,2023-05-01,36.6397,127.3969,36.6399,127.3846,1.098
1,211,인제,2025-07-30,38.0599,128.1671,38.0599,128.1681,0.088
2,217,정선군,2023-07-01,37.3815,128.6459,37.3773,128.6735,2.483
3,217,정선군,2025-12-17,37.3773,128.6735,37.3807,128.6731,0.380
4,259,강진군,2021-02-25,34.6289,126.7672,34.6446,126.7841,2.332


In [8]:
transition_dates = {
    181: "2023-05-01",
    211: "2025-07-30",
    217: "2023-07-01",
    259: "2021-02-25"
}

for station_id, date in transition_dates.items():

    date = pd.Timestamp(date)

    print(f"\n{'=' * 40}")
    print(f"지점번호: {station_id}")
    print(f"변경일: {date.date()}")

    result = temperature_df[
        (temperature_df["지점"] == station_id) &
        (temperature_df["일시"].between(
            (date - pd.Timedelta(days=3)).strftime("%Y-%m-%d"),
            (date + pd.Timedelta(days=3)).strftime("%Y-%m-%d")
        ))
    ]

    display(
        result[
            ["지점", "지점명", "일시",
             "최저기온(°C)", "최고기온(°C)"]
        ]
    )


지점번호: 181
변경일: 2023-05-01


,지점,지점명,일시,최저기온(°C),최고기온(°C)
74816,181,서청주,2023-05-01,6.9,21.0
74817,181,서청주,2023-05-02,3.7,26.3
74818,181,서청주,2023-05-03,10.0,26.5
74819,181,서청주,2023-05-04,13.2,27.3



지점번호: 211
변경일: 2025-07-30


,지점,지점명,일시,최저기온(°C),최고기온(°C)
92063,211,인제,2025-07-27,22.2,34.8
92064,211,인제,2025-07-28,21.9,29.0
92065,211,인제,2025-07-30,23.4,34.5
92066,211,인제,2025-07-31,23.9,32.4
92067,211,인제,2025-08-01,22.5,34.4
92068,211,인제,2025-08-02,23.9,33.4



지점번호: 217
변경일: 2023-07-01


,지점,지점명,일시,최저기온(°C),최고기온(°C)
96776,217,정선군,2023-07-01,20.5,34.5
96777,217,정선군,2023-07-02,20.6,33.1
96778,217,정선군,2023-07-04,21.6,31.8



지점번호: 259
변경일: 2021-02-25


,지점,지점명,일시,최저기온(°C),최고기온(°C)
132420,259,강진군,2021-02-22,3.1,19.9
132421,259,강진군,2021-02-23,-0.4,10.0
132422,259,강진군,2021-02-24,-1.8,12.5
132423,259,강진군,2021-02-25,2.5,10.4
132424,259,강진군,2021-02-26,5.8,14.6
132425,259,강진군,2021-02-27,5.8,11.8
132426,259,강진군,2021-02-28,6.9,11.7


In [10]:
# 관측소별 시작일 기준 정렬
station_coordinate_df = (
    station_period_df
    .sort_values(["지점", "시작일"])
    .drop_duplicates(subset=["지점"], keep="last")
    .copy()
)

# 기존 기온 데이터의 시도 정보 결합
station_coordinate_df = station_coordinate_df.merge(
    station_df[["지점", "시도"]],
    on="지점",
    how="left"
)

# 필요한 컬럼만 추출
station_coordinate_df = station_coordinate_df[
    ["지점", "지점명", "시도", "위도", "경도",
     "시작일", "종료일", "지점주소"]
].reset_index(drop=True)

# 좌표 숫자형 변환
station_coordinate_df["위도"] = pd.to_numeric(
    station_coordinate_df["위도"], errors="coerce"
)

station_coordinate_df["경도"] = pd.to_numeric(
    station_coordinate_df["경도"], errors="coerce"
)

print("대표 관측소 수:", len(station_coordinate_df))
print("중복 지점번호:", station_coordinate_df["지점"].duplicated().sum())
print("시도 결측:", station_coordinate_df["시도"].isna().sum())
print("위도 결측:", station_coordinate_df["위도"].isna().sum())
print("경도 결측:", station_coordinate_df["경도"].isna().sum())

display(station_coordinate_df.head(10))

대표 관측소 수: 97
중복 지점번호: 0
시도 결측: 0
위도 결측: 0
경도 결측: 0


,지점,지점명,시도,위도,경도,시작일,종료일,지점주소
0,90,속초,강원특별자치도,38.2509,128.5647,1968-01-01,NaT,강원특별자치도 고성군 토성면 봉포리
1,93,북춘천,강원특별자치도,37.9474,127.7544,2016-10-01,NaT,강원특별자치도 춘천시 신북읍 산천리
2,95,철원,강원특별자치도,38.1479,127.3042,1988-01-01,NaT,강원특별자치도 철원군 갈말읍 군탄리
3,98,동두천,경기도,37.9019,127.0607,1998-02-01,NaT,경기도 동두천시 생연동
4,99,파주,경기도,37.8859,126.7665,2013-10-22,NaT,경기도 파주시 문산읍 운천리
5,100,대관령,강원특별자치도,37.6771,128.7183,2006-11-07,NaT,(산지)강원특별자치도 평창군 대관령면 횡계리
6,101,춘천,강원특별자치도,37.9026,127.7357,1966-01-01,NaT,강원특별자치도 춘천시 우두동
7,102,백령도,인천광역시,37.9740,124.7124,2018-05-01,NaT,인천광역시 옹진군 백령면 진촌리
8,104,북강릉,강원특별자치도,37.8046,128.8554,2008-07-28,NaT,강원특별자치도 강릉시 사천면 방동리
9,105,강릉,강원특별자치도,37.7515,128.8910,1911-10-03,NaT,강원특별자치도 강릉시 용강동


In [11]:
output_path = (
    PROCESSED_DIR / "기상관측소_좌표_정제.csv"
)

station_coordinate_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)

저장 완료: c:\Users\expri\Desktop\SafeWeather-AI\data\processed\기상관측소_좌표_정제.csv


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

# 시군구 대표 좌표
region_df = pd.read_csv(
    PROCESSED_DIR / "시군구_대표좌표_정제.csv"
)

# 기상관측소 좌표
station_coordinate_df = pd.read_csv(
    PROCESSED_DIR / "기상관측소_좌표_정제.csv"
)

print("시군구 데이터:", region_df.shape)
print("기상관측소 데이터:", station_coordinate_df.shape)

display(region_df.head())
display(station_coordinate_df.head())

시군구 데이터: (229, 4)
기상관측소 데이터: (97, 8)


,시도,시군구,경도,위도
0,강원특별자치도,강릉시,128.883209,37.709604
1,강원특별자치도,고성군,128.358361,38.401009
2,강원특별자치도,동해시,129.045397,37.515759
3,강원특별자치도,삼척시,129.173724,37.257882
4,강원특별자치도,속초시,128.515038,38.175014


,지점,지점명,시도,위도,경도,시작일,종료일,지점주소
0,90,속초,강원특별자치도,38.2509,128.5647,1968-01-01,NaN,강원특별자치도 고성군 토성면 봉포리
1,93,북춘천,강원특별자치도,37.9474,127.7544,2016-10-01,NaN,강원특별자치도 춘천시 신북읍 산천리
2,95,철원,강원특별자치도,38.1479,127.3042,1988-01-01,NaN,강원특별자치도 철원군 갈말읍 군탄리
3,98,동두천,경기도,37.9019,127.0607,1998-02-01,NaN,경기도 동두천시 생연동
4,99,파주,경기도,37.8859,126.7665,2013-10-22,NaN,경기도 파주시 문산읍 운천리


In [2]:
print("[시군구]")
print("지역 중복:", region_df.duplicated(["시도", "시군구"]).sum())
print("위도 결측:", region_df["위도"].isna().sum())
print("경도 결측:", region_df["경도"].isna().sum())

print("\n[기상관측소]")
print("관측소 중복:", station_coordinate_df["지점"].duplicated().sum())
print("위도 결측:", station_coordinate_df["위도"].isna().sum())
print("경도 결측:", station_coordinate_df["경도"].isna().sum())

[시군구]
지역 중복: 0
위도 결측: 0
경도 결측: 0

[기상관측소]
관측소 중복: 0
위도 결측: 0
경도 결측: 0


In [3]:
# 지구 반지름 (km)
EARTH_RADIUS = 6371.0

# 시군구 좌표를 라디안으로 변환
region_lat = np.radians(region_df["위도"].to_numpy())
region_lon = np.radians(region_df["경도"].to_numpy())

# 기상관측소 좌표를 라디안으로 변환
station_lat = np.radians(station_coordinate_df["위도"].to_numpy())
station_lon = np.radians(station_coordinate_df["경도"].to_numpy())

# 위도와 경도 차이 계산
lat_diff = station_lat[np.newaxis, :] - region_lat[:, np.newaxis]
lon_diff = station_lon[np.newaxis, :] - region_lon[:, np.newaxis]

# 하버사인 공식
a = (
    np.sin(lat_diff / 2) ** 2
    + np.cos(region_lat[:, np.newaxis])
    * np.cos(station_lat[np.newaxis, :])
    * np.sin(lon_diff / 2) ** 2
)

c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# 거리 행렬 (229개 지역 × 97개 관측소)
distance_matrix = EARTH_RADIUS * c

print("거리 행렬 크기:", distance_matrix.shape)
print("최소 거리:", round(distance_matrix.min(), 2), "km")
print("최대 거리:", round(distance_matrix.max(), 2), "km")

거리 행렬 크기: (229, 97)
최소 거리: 0.14 km
최대 거리: 632.46 km


In [4]:
# 각 시군구에서 가장 가까운 관측소의 인덱스
nearest_indices = np.argmin(distance_matrix, axis=1)

# 가장 가까운 관측소 정보 추출
nearest_stations = station_coordinate_df.iloc[
    nearest_indices
].reset_index(drop=True)

# 최종 매칭 데이터 생성
region_station_df = region_df.copy()

region_station_df["관측소_지점"] = nearest_stations["지점"].values
region_station_df["관측소명"] = nearest_stations["지점명"].values
region_station_df["관측소_시도"] = nearest_stations["시도"].values
region_station_df["거리_km"] = distance_matrix[
    np.arange(len(region_df)),
    nearest_indices
]

# 거리 기준 정렬
region_station_df = region_station_df.sort_values(
    "거리_km",
    ascending=False
).reset_index(drop=True)

display(region_station_df.head(20))

,시도,시군구,경도,위도,관측소_지점,관측소명,관측소_시도,거리_km
0,충청북도,음성군,127.552274,36.994613,203,이천,경기도,30.556337
1,경기도,가평군,127.407659,37.811246,101,춘천,강원특별자치도,30.538766
2,경기도,안성시,127.305900,37.024761,232,천안,충청남도,29.218674
3,경상북도,영양군,129.135460,36.686126,276,청송군,경상북도,29.183307
4,전북특별자치도,무주군,127.716871,35.928336,238,금산,충청남도,28.905083
5,충청남도,아산시,126.976776,36.795193,232,천안,충청남도,28.383790
6,강원특별자치도,홍천군,128.184416,37.747630,212,홍천,강원특별자치도,27.673259
7,강원특별자치도,평창군,128.441486,37.545291,217,정선군,강원특별자치도,27.437941
8,강원특별자치도,양양군,128.622440,38.011551,90,속초,강원특별자치도,27.089328
9,경상북도,청도군,128.883494,35.706844,288,밀양,경상남도,27.059340


In [5]:
print("매칭 지역 수:", len(region_station_df))
print("관측소 결측치:", region_station_df["관측소명"].isna().sum())

print("\n가장 가까운 관측소까지의 거리 통계")
display(region_station_df["거리_km"].describe().round(2))

print("\n가장 가까운 관측소가 가장 먼 지역 TOP 10")
display(
    region_station_df[
        ["시도", "시군구", "관측소명", "거리_km"]
    ].head(10)
)

print(
    "\n다른 시도 관측소에 연결된 지역 수:",
    (region_station_df["시도"] != region_station_df["관측소_시도"]).sum()
)

매칭 지역 수: 229
관측소 결측치: 0

가장 가까운 관측소까지의 거리 통계


count    229.00
mean      11.77
std        7.48
min        0.14
25%        5.55
50%       10.39
75%       17.06
max       30.56
Name: 거리_km, dtype: float64


가장 가까운 관측소가 가장 먼 지역 TOP 10


,시도,시군구,관측소명,거리_km
0,충청북도,음성군,이천,30.556337
1,경기도,가평군,춘천,30.538766
2,경기도,안성시,천안,29.218674
3,경상북도,영양군,청송군,29.183307
4,전북특별자치도,무주군,금산,28.905083
5,충청남도,아산시,천안,28.383790
6,강원특별자치도,홍천군,홍천,27.673259
7,강원특별자치도,평창군,정선군,27.437941
8,강원특별자치도,양양군,속초,27.089328
9,경상북도,청도군,밀양,27.059340



다른 시도 관측소에 연결된 지역 수: 52


In [6]:
output_path = PROCESSED_DIR / "시군구_기상관측소_매칭_정제.csv"

region_station_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)

저장 완료: c:\Users\expri\Desktop\SafeWeather-AI\data\processed\시군구_기상관측소_매칭_정제.csv


In [7]:
# 폭염일수 데이터
heatwave_df = pd.read_csv(
    PROCESSED_DIR / "폭염일수_정제.csv"
)

# 한파일수 데이터
coldwave_df = pd.read_csv(
    PROCESSED_DIR / "한파일수_정제.csv"
)

print("[폭염일수]")
print("크기:", heatwave_df.shape)
display(heatwave_df.head())

print("\n[한파일수]")
print("크기:", coldwave_df.shape)
display(coldwave_df.head())

[폭염일수]
크기: (481, 9)


,지점,지점명,시도,연도,폭염일수,관측일수,여름관측일수,여름부족일수,여름관측_주의
0,90,속초,강원특별자치도,2021,4,365,122,0,False
1,90,속초,강원특별자치도,2022,6,365,122,0,False
2,90,속초,강원특별자치도,2023,13,365,122,0,False
3,90,속초,강원특별자치도,2024,16,366,122,0,False
4,90,속초,강원특별자치도,2025,14,365,122,0,False



[한파일수]
크기: (481, 9)


,지점,지점명,시도,연도,한파일수,관측일수,겨울관측일수,겨울부족일수,겨울관측_주의
0,90,속초,강원특별자치도,2021,7,365,151,0,False
1,90,속초,강원특별자치도,2022,1,365,151,0,False
2,90,속초,강원특별자치도,2023,4,365,151,0,False
3,90,속초,강원특별자치도,2024,2,366,152,0,False
4,90,속초,강원특별자치도,2025,0,365,151,0,False


In [8]:
# 병합 기준 중복 확인
print("폭염 데이터 중복:", 
      heatwave_df.duplicated(["지점", "연도"]).sum())

print("한파 데이터 중복:", 
      coldwave_df.duplicated(["지점", "연도"]).sum())


# 폭염 데이터와 한파 데이터 병합
climate_df = pd.merge(
    heatwave_df,
    coldwave_df,
    on=["지점", "지점명", "시도", "연도"],
    how="inner",
    validate="one_to_one"
)

print("\n병합 결과:", climate_df.shape)

display(
    climate_df[
        ["지점", "지점명", "시도", "연도",
         "폭염일수", "한파일수",
         "여름관측_주의", "겨울관측_주의"]
    ].head(10)
)

폭염 데이터 중복: 0
한파 데이터 중복: 0

병합 결과: (481, 14)


,지점,지점명,시도,연도,폭염일수,한파일수,여름관측_주의,겨울관측_주의
0,90,속초,강원특별자치도,2021,4,7,False,False
1,90,속초,강원특별자치도,2022,6,1,False,False
2,90,속초,강원특별자치도,2023,13,4,False,False
3,90,속초,강원특별자치도,2024,16,2,False,False
4,90,속초,강원특별자치도,2025,14,0,False,False
5,93,북춘천,강원특별자치도,2021,21,28,False,False
6,93,북춘천,강원특별자치도,2022,12,45,False,False
7,93,북춘천,강원특별자치도,2023,24,21,False,False
8,93,북춘천,강원특별자치도,2024,32,14,False,False
9,93,북춘천,강원특별자치도,2025,30,24,False,False


In [9]:
# 시군구별 기후 데이터 연결
region_climate_df = pd.merge(
    region_station_df,
    climate_df,
    left_on="관측소_지점",
    right_on="지점",
    how="left",
    validate="many_to_many",
    suffixes=("", "_기상")
)

# 불필요한 중복 컬럼 제거
region_climate_df = region_climate_df.drop(
    columns=["지점", "지점명", "시도_기상"]
)

# 지역 및 연도 기준 정렬
region_climate_df = region_climate_df.sort_values(
    ["시도", "시군구", "연도"]
).reset_index(drop=True)

print("최종 데이터 크기:", region_climate_df.shape)
print("지역 수:", region_climate_df[["시도", "시군구"]].drop_duplicates().shape[0])
print("연도 수:", region_climate_df["연도"].nunique())

display(region_climate_df.head(10))

최종 데이터 크기: (1137, 19)
지역 수: 229
연도 수: 5


,시도,시군구,경도,위도,관측소_지점,관측소명,관측소_시도,거리_km,연도,폭염일수,관측일수_x,여름관측일수,여름부족일수,여름관측_주의,한파일수,관측일수_y,겨울관측일수,겨울부족일수,겨울관측_주의
0,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2021,11,365,122,0,False,4,365,151,0,False
1,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2022,16,365,122,0,False,0,365,151,0,False
2,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2023,21,365,122,0,False,2,365,151,0,False
3,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2024,35,366,122,0,False,0,366,152,0,False
4,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2025,43,365,122,0,False,0,365,151,0,False
5,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2021,4,365,122,0,False,7,365,151,0,False
6,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2022,6,365,122,0,False,1,365,151,0,False
7,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2023,13,365,122,0,False,4,365,151,0,False
8,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2024,16,366,122,0,False,2,366,152,0,False
9,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2025,14,365,122,0,False,0,365,151,0,False


In [10]:
# 지역별 연도 개수 확인
year_count = (
    region_climate_df
    .groupby(["시도", "시군구"])["연도"]
    .nunique()
)

print("5개 연도 모두 보유한 지역:", (year_count == 5).sum())
print("5개 연도 미만인 지역:", (year_count < 5).sum())


# 누락된 지역 확인
missing_regions = (
    year_count[year_count < 5]
    .reset_index(name="연도_개수")
)

display(missing_regions)

5개 연도 모두 보유한 지역: 225
5개 연도 미만인 지역: 4


,시도,시군구,연도_개수
0,부산광역시,강서구,3
1,부산광역시,동래구,3
2,부산광역시,북구,3
3,부산광역시,사상구,3


In [11]:
# 229개 지역 × 5개 연도 전체 조합 생성
all_years = [2021, 2022, 2023, 2024, 2025]

expected_df = (
    region_station_df
    .assign(key=1)
    .merge(
        pd.DataFrame({"연도": all_years, "key": 1}),
        on="key"
    )
    .drop(columns="key")
)

# 실제 기후 데이터와 비교
check_df = expected_df.merge(
    climate_df[["지점", "연도"]],
    left_on=["관측소_지점", "연도"],
    right_on=["지점", "연도"],
    how="left",
    indicator=True
)

missing_climate = check_df[
    check_df["_merge"] == "left_only"
]

print("누락된 지역-연도 행:", len(missing_climate))

display(
    missing_climate[
        ["시도", "시군구", "관측소_지점", "관측소명", "연도"]
    ]
)

누락된 지역-연도 행: 8


,시도,시군구,관측소_지점,관측소명,연도
525,부산광역시,동래구,296,북부산,2021
526,부산광역시,동래구,296,북부산,2022
730,부산광역시,강서구,296,북부산,2021
731,부산광역시,강서구,296,북부산,2022
780,부산광역시,사상구,296,북부산,2021
781,부산광역시,사상구,296,북부산,2022
845,부산광역시,북구,296,북부산,2021
846,부산광역시,북구,296,북부산,2022


In [12]:
# 북부산 관측소의 기후 데이터
display(
    climate_df[
        climate_df["지점"] == 296
    ][
        ["지점", "지점명", "연도", "폭염일수", "한파일수"]
    ].sort_values("연도")
)

,지점,지점명,연도,폭염일수,한파일수
478,296,북부산,2023,14,1
479,296,북부산,2024,44,0
480,296,북부산,2025,42,0


In [14]:
# 기상청 관측지점 메타데이터 불러오기
station_info_path = ROOT_DIR / "data" / "raw" / "META_관측지점정보.csv"

try:
    station_info_df = pd.read_csv(
        station_info_path,
        encoding="utf-8-sig"
    )
except UnicodeDecodeError:
    station_info_df = pd.read_csv(
        station_info_path,
        encoding="cp949"
    )

# 북부산 관측소 정보 확인
display(
    station_info_df[
        station_info_df["지점"] == 296
    ][
        ["지점", "지점명", "시작일", "종료일", "위도", "경도"]
    ]
)

,지점,지점명,시작일,종료일,위도,경도
147,296,북부산,2025-04-25,NaN,35.2178,128.9602


In [16]:
temperature_df = pd.read_csv(
    PROCESSED_DIR / "기상_일별_최고최저기온_정제.csv"
)

print("데이터 크기:", temperature_df.shape)
print("컬럼:", temperature_df.columns.tolist())

데이터 크기: (175402, 6)
컬럼: ['지점', '지점명', '시도', '일시', '최저기온(°C)', '최고기온(°C)']


In [17]:
busan_north_temp = temperature_df[
    temperature_df["지점"] == 296
].copy()

print("데이터 행 수:", len(busan_north_temp))

print("\n관측소 이름:")
display(busan_north_temp["지점명"].value_counts())

print("\n연도별 관측 현황:")
display(
    busan_north_temp.assign(
        연도=pd.to_datetime(
            busan_north_temp["일시"]
        ).dt.year
    )
    .groupby("연도")
    .agg(
        관측일수=("일시", "count"),
        시작일=("일시", "min"),
        종료일=("일시", "max")
    )
)

데이터 행 수: 1084

관측소 이름:


지점명
북부산    1084
Name: count, dtype: int64


연도별 관측 현황:


,관측일수,시작일,종료일
연도,,,
2023,355,2023-01-11,2023-12-31
2024,364,2024-01-01,2024-12-31
2025,365,2025-01-01,2025-12-31


In [18]:
# 5개 연도 자료가 모두 존재하는 관측소
complete_station_ids = (
    climate_df.groupby("지점")["연도"]
    .nunique()
)

complete_station_ids = complete_station_ids[
    complete_station_ids == 5
].index

# 북부산 제외
candidate_stations = station_coordinate_df[
    station_coordinate_df["지점"].isin(complete_station_ids)
    & (station_coordinate_df["지점"] != 296)
].copy()

# 북부산을 가장 가까운 관측소로 사용했던 지역
target_regions = region_station_df[
    region_station_df["관측소_지점"] == 296
].copy()

results = []

for _, region in target_regions.iterrows():

    lat1 = np.radians(region["위도"])
    lon1 = np.radians(region["경도"])

    lat2 = np.radians(candidate_stations["위도"].to_numpy())
    lon2 = np.radians(candidate_stations["경도"].to_numpy())

    lat_diff = lat2 - lat1
    lon_diff = lon2 - lon1

    a = (
        np.sin(lat_diff / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(lon_diff / 2) ** 2
    )

    distances = 6371.0 * (
        2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    )

    candidates = candidate_stations.copy()
    candidates["거리_km"] = distances

    top_candidates = candidates.nsmallest(3, "거리_km")

    for _, station in top_candidates.iterrows():
        results.append({
            "시군구": region["시군구"],
            "후보_관측소": station["지점명"],
            "관측소_지점": station["지점"],
            "거리_km": round(station["거리_km"], 2)
        })

alternative_df = pd.DataFrame(results)

display(alternative_df)

,시군구,후보_관측소,관측소_지점,거리_km
0,동래구,부산,159,11.99
1,동래구,양산시,257,12.60
2,동래구,김해시,253,17.40
3,강서구,김해시,253,8.37
4,강서구,부산,159,11.94
5,강서구,양산시,257,19.05
6,사상구,부산,159,7.39
7,사상구,김해시,253,11.67
8,사상구,양산시,257,16.71
9,북구,양산시,257,8.40


In [19]:
candidate_ids = [159, 253, 257]

# 연도별 폭염·한파 및 관측 현황
candidate_check = climate_df[
    climate_df["지점"].isin(candidate_ids)
][
    [
        "지점", "지점명", "연도",
        "폭염일수", "한파일수",
        "관측일수_x", "여름관측일수",
        "여름관측_주의",
        "관측일수_y", "겨울관측일수",
        "겨울관측_주의"
    ]
].sort_values(["지점", "연도"])

display(candidate_check)

,지점,지점명,연도,폭염일수,한파일수,관측일수_x,여름관측일수,여름관측_주의,관측일수_y,겨울관측일수,겨울관측_주의
160,159,부산,2021,3,1,365,122,False,365,151,False
161,159,부산,2022,1,0,365,122,False,365,151,False
162,159,부산,2023,9,1,365,122,False,365,151,False
163,159,부산,2024,22,0,366,122,False,366,152,False
164,159,부산,2025,16,0,365,122,False,365,151,False
338,253,김해시,2021,17,2,365,122,False,365,151,False
339,253,김해시,2022,12,0,365,122,False,365,151,False
340,253,김해시,2023,19,1,365,122,False,365,151,False
341,253,김해시,2024,51,0,366,122,False,366,152,False
342,253,김해시,2025,43,0,365,122,False,365,151,False


In [20]:
# 기존 매칭 데이터 복사
region_station_final = region_station_df.copy()

# 지역별 대체 관측소 번호
replacement_map = {
    "동래구": 159,
    "강서구": 253,
    "사상구": 159,
    "북구": 257
}

# 대체 관측소 정보 반영
for region, station_id in replacement_map.items():

    mask = (
        (region_station_final["시도"] == "부산광역시")
        & (region_station_final["시군구"] == region)
    )

    station = station_coordinate_df[
        station_coordinate_df["지점"] == station_id
    ].iloc[0]

    region_station_final.loc[mask, "관측소_지점"] = station_id
    region_station_final.loc[mask, "관측소명"] = station["지점명"]
    region_station_final.loc[mask, "관측소_시도"] = station["시도"]

    region_station_final.loc[mask, "거리_km"] = (
        alternative_df[
            (alternative_df["시군구"] == region)
            & (alternative_df["관측소_지점"] == station_id)
        ]["거리_km"].iloc[0]
    )

# 변경된 지역 확인
display(
    region_station_final[
        region_station_final["시군구"].isin(replacement_map.keys())
    ][
        ["시도", "시군구", "관측소_지점", "관측소명", "거리_km"]
    ]
)

,시도,시군구,관측소_지점,관측소명,거리_km
86,서울특별시,강서구,108,서울,12.997635
105,부산광역시,동래구,159,부산,11.990000
129,대구광역시,북구,143,대구,8.790697
146,부산광역시,강서구,253,김해시,8.370000
156,부산광역시,사상구,159,부산,7.390000
167,울산광역시,북구,152,울산,5.939206
169,부산광역시,북구,257,양산시,8.400000
212,전남광주통합특별시,북구,156,광주,2.478924


In [21]:
# 필요한 기후 데이터 컬럼만 선택
climate_selected = climate_df[
    [
        "지점", "연도",
        "폭염일수", "관측일수_x",
        "여름관측일수", "여름부족일수",
        "여름관측_주의",
        "한파일수", "관측일수_y",
        "겨울관측일수", "겨울부족일수",
        "겨울관측_주의"
    ]
].copy()

# 시군구별 기후 데이터 병합
region_climate_final = pd.merge(
    region_station_final,
    climate_selected,
    left_on="관측소_지점",
    right_on="지점",
    how="left"
)

# 중복 관측소 번호 제거
region_climate_final = region_climate_final.drop(
    columns=["지점"]
)

# 지역 및 연도 기준 정렬
region_climate_final = region_climate_final.sort_values(
    ["시도", "시군구", "연도"]
).reset_index(drop=True)

print("최종 데이터 크기:", region_climate_final.shape)
print("지역 수:", region_climate_final[["시도", "시군구"]].drop_duplicates().shape[0])
print("연도 수:", region_climate_final["연도"].nunique())

display(region_climate_final.head(10))

최종 데이터 크기: (1145, 19)
지역 수: 229
연도 수: 5


,시도,시군구,경도,위도,관측소_지점,관측소명,관측소_시도,거리_km,연도,폭염일수,관측일수_x,여름관측일수,여름부족일수,여름관측_주의,한파일수,관측일수_y,겨울관측일수,겨울부족일수,겨울관측_주의
0,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2021,11,365,122,0,False,4,365,151,0,False
1,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2022,16,365,122,0,False,0,365,151,0,False
2,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2023,21,365,122,0,False,2,365,151,0,False
3,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2024,35,366,122,0,False,0,366,152,0,False
4,강원특별자치도,강릉시,128.883209,37.709604,105,강릉,강원특별자치도,4.708752,2025,43,365,122,0,False,0,365,151,0,False
5,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2021,4,365,122,0,False,7,365,151,0,False
6,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2022,6,365,122,0,False,1,365,151,0,False
7,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2023,13,365,122,0,False,4,365,151,0,False
8,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2024,16,366,122,0,False,2,366,152,0,False
9,강원특별자치도,고성군,128.358361,38.401009,90,속초,강원특별자치도,24.547473,2025,14,365,122,0,False,0,365,151,0,False


In [22]:
# 1. 전체 데이터 크기
print("데이터 크기:", region_climate_final.shape)

# 2. 지역 수
print(
    "지역 수:",
    region_climate_final[["시도", "시군구"]]
    .drop_duplicates().shape[0]
)

# 3. 연도별 데이터 개수
print("\n[연도별 데이터 개수]")
display(
    region_climate_final.groupby("연도").size()
)

# 4. 지역-연도 중복 확인
print(
    "\n지역-연도 중복:",
    region_climate_final.duplicated(
        ["시도", "시군구", "연도"]
    ).sum()
)

# 5. 결측치 확인
print("\n[결측치]")
display(
    region_climate_final.isna().sum()
)

# 6. 관측 주의 여부
print("\n[관측 주의 여부]")
print(
    "여름 관측 주의:",
    region_climate_final["여름관측_주의"].sum()
)
print(
    "겨울 관측 주의:",
    region_climate_final["겨울관측_주의"].sum()
)

데이터 크기: (1145, 19)
지역 수: 229

[연도별 데이터 개수]


연도
2021    229
2022    229
2023    229
2024    229
2025    229
dtype: int64


지역-연도 중복: 0

[결측치]


시도         0
시군구        0
경도         0
위도         0
관측소_지점     0
관측소명       0
관측소_시도     0
거리_km      0
연도         0
폭염일수       0
관측일수_x     0
여름관측일수     0
여름부족일수     0
여름관측_주의    0
한파일수       0
관측일수_y     0
겨울관측일수     0
겨울부족일수     0
겨울관측_주의    0
dtype: int64


[관측 주의 여부]
여름 관측 주의: 1
겨울 관측 주의: 0


In [23]:
summer_warning = region_climate_final[
    region_climate_final["여름관측_주의"] == True
]

display(
    summer_warning[
        [
            "시도", "시군구", "연도",
            "관측소명", "거리_km",
            "폭염일수", "여름관측일수",
            "여름부족일수", "여름관측_주의"
        ]
    ]
)

,시도,시군구,연도,관측소명,거리_km,폭염일수,여름관측일수,여름부족일수,여름관측_주의
915,전남광주통합특별시,진도군,2021,진도군,0.476969,3,114,8,True


In [24]:
# 최종 기후 데이터 저장
output_path = PROCESSED_DIR / "시군구_기후데이터_정제.csv"

region_climate_final.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)

저장 완료: c:\Users\expri\Desktop\SafeWeather-AI\data\processed\시군구_기후데이터_정제.csv
